In [13]:
# ----------------------------------
# 
# postprocess flux data 
# 
# NOTE: This script uses the batch
# pattern that relies on a batch.csv
# file (not a dir of dicts)
# 
# ----------------------------------
import os

import numpy as np
import pandas as pd

# --- read in cdr calculation functions  
from ew_workflows import cdr_fxns_postproc as cfp
from ew_workflows import cflx_proc as flxs
# ---

# --- Decide which CDR calculations to perform
cdr_calc_list = ["co2_flx",       
                 "camg_flx",
                 "totcat_flx",
                 # "carbalk_flx", # (turn this back on when the runs are re-run with fixed cflx file)
                 "rockdiss"]
# ---

In [25]:
# ----------------------------------
# [ SETTINGS ]
EXP_GROUP = "lags_singleApp"
EXP_SET = f"{EXP_GROUP}_gbas_sites"
multiyear = True
dustsp = "gbas"
runtype_time = "ann_ltm"
runtype = "field"
extra_tag = "v0"
pref = f"{EXP_SET}_{runtype_time}_{extra_tag}"
batchname = pref + ".csv"
# data structure
ctrl_conditions = {"dustrate": 0} 
collapse_cols = [ # [ list or None ] # params that are analyzed one at a time, rather than all combinations (we don't want an xr dimension for each param)
    # "dustrad", "secondary_min_rule", "poro_updated",
] 
dfin_cols_to_keep = [   # become dimensions in later xr dataset
    "dustrate_ton_ha_yr", "dustrate_2nd", "site",
] 
dfin_cols_to_keep_time = dfin_cols_to_keep + ["time"]
# ----------------------------------

# --- read in the batch .csv
batchdir = "s3://carbonplan-carbon-removal/ew-workflows-data/scepter/batch/"
dfin = pd.read_csv(os.path.join(batchdir, batchname))

# ---
# groundwork
outdir = f"s3://carbonplan-carbon-removal/SCEPTER/scepter_output/{EXP_GROUP}/{EXP_SET}"

In [26]:
outdir

's3://carbonplan-carbon-removal/SCEPTER/scepter_output/lags_singleApp/lags_singleApp_gbas_sites'

In [28]:
# --- add column for the full run ID
if multiyear:
    dfin["newrun_id_full"] = dfin["newrun_id"] + f"_composite_{runtype}"
else:
    dfin['newrun_id_full'] = dfin['newrun_id'] + "_" + dfin['dustsp'] + "_" + runtype + "_tau"+dfin["duration"].astype(float).astype(str).str.replace(".", "p")  # (duration has to be turned into float first because otherwise we miss the decimal pt)

# identify the control run(s)
dfin = cfp.find_control_runs(dfin, ctrl_conditions)
# add a column for the dustrate in ton_ha_yr
if "dustrate" in dfin.columns:
    dfin["dustrate_ton_ha_yr"] = dfin["dustrate"] / 100 

# add dustrad if we only have psdrain_meanRad
if "dustrad" not in dfin.columns:
    dfin['dustrad'] = dfin['psdrain_meanRad'] * 1e6 # [ convert m to um ]

# --- extract sites
sitenames = dfin['site'].unique()

# --- create the collapsed col
if collapse_cols:
    dfin["collapsed_inputs"] = dfin[collapse_cols].apply(
       lambda row: " | ".join(f"{col}={row[col]}" for col in collapse_cols),
        axis=1
    )
    # update the ds coords
    dfin_cols_to_keep = dfin_cols_to_keep + ["collapsed_inputs"]
    dfin_cols_to_keep_time = dfin_cols_to_keep_time + ["collapsed_inputs"]


## Flux postprocessing

#### CDR fluxes

In [30]:
# --- read in the postprocessed flux data 
# [ integrated flux ]
flx_dict_int = cfp.read_postproc_flux(
    dfin, outdir, cdr_calc_list, dfin_cols_to_keep, 
    flx_type="int_flx", rockdiss_feedstock=None
)
# [ flux ]
flx_dict = cfp.read_postproc_flux(
    dfin, outdir, cdr_calc_list, dfin_cols_to_keep, 
    flx_type="flx", rockdiss_feedstock=None
)

solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss
solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss


#### Other fluxes (not preprocessed)
TODO --- read in ALK fluxes, use infiltration to convert to concentrations, then compare to bottomwater ph? Or just compute alk profiles from charge balance (@ yoshi..?)

In [2]:
import pandas as pd
pd.read_pickle('s3://carbonplan-carbon-removal/SCEPTER/scepter_output/lags_singleApp/lags_singleApp_gbas_sites/lags_singleApp_gbas_sites_ann_ltm_v0_264_cornbelt_ann_ltm_app_0p0_psize_75_0_composite_field/postproc_flxs/carbAlk_flxs.pkl')

,time,calkflx_tflx,calkflx_adv,calkflx_tot,co2pot_adv_tonHaYr_sil,co2pot_adv_tonHaYr_cc,co2pot_tot_tonHaYr_sil,co2pot_tot_tonHaYr_cc,units,flx_type,runname,var,co2pot_adv_tonHa_sil,co2pot_adv_tonHa_cc,co2pot_tot_tonHa_sil,co2pot_tot_tonHa_cc
0,0.000009,26.459542,-0.137529,26.322013,-0.121053,-0.060526,23.168636,11.584318,mol m-2 yr,flx,lags_singleApp_gbas_sites_ann_ltm_v0_264_cornb...,ALK,NaN,NaN,NaN,NaN
1,0.000026,25.292493,-0.137529,25.154964,-0.121053,-0.060526,22.141399,11.070700,mol m-2 yr,flx,lags_singleApp_gbas_sites_ann_ltm_v0_264_cornb...,ALK,NaN,NaN,NaN,NaN
2,0.000084,22.357463,-0.137529,22.219935,-0.121053,-0.060526,19.557987,9.778993,mol m-2 yr,flx,lags_singleApp_gbas_sites_ann_ltm_v0_264_cornb...,ALK,NaN,NaN,NaN,NaN
3,0.000251,17.837340,-0.137529,17.699811,-0.121053,-0.060526,15.579374,7.789687,mol m-2 yr,flx,lags_singleApp_gbas_sites_ann_ltm_v0_264_cornb...,ALK,NaN,NaN,NaN,NaN
4,0.000418,15.202727,-0.137528,15.065198,-0.121053,-0.060526,13.260387,6.630194,mol m-2 yr,flx,lags_singleApp_gbas_sites_ann_ltm_v0_264_cornb...,ALK,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,12.666668,-0.170260,1.763790,1.593530,NaN,NaN,NaN,NaN,mol m-2,int_flx,lags_singleApp_gbas_sites_ann_ltm_v0_264_cornb...,ALK,1.552488,0.776244,1.402625,0.701312
252,13.250000,-0.173858,1.714043,1.540186,NaN,NaN,NaN,NaN,mol m-2,int_flx,lags_singleApp_gbas_sites_ann_ltm_v0_264_cornb...,ALK,1.508701,0.754350,1.355671,0.677836
253,13.833334,-0.177369,1.658160,1.480791,NaN,NaN,NaN,NaN,mol m-2,int_flx,lags_singleApp_gbas_sites_ann_ltm_v0_264_cornb...,ALK,1.459513,0.729756,1.303392,0.651696
254,14.416668,-0.180539,1.596553,1.416014,NaN,NaN,NaN,NaN,mol m-2,int_flx,lags_singleApp_gbas_sites_ann_ltm_v0_264_cornb...,ALK,1.405286,0.702643,1.246375,0.623188


In [ ]:
runname = dfin['newrun_id_full'].values[0]
var_dict = {
    'alk': ['flx_co2sp', 'ALK']
}

# flxs.get_data(outdir, runname, )

'lags_singleApp_gbas_sites_ann_ltm_v0_264_cornbelt_ann_ltm_app_0p0_psize_75_0_composite_field'